In [1]:
import json
import os
import sys

sys.path.insert(0, os.path.abspath(".."))

from datetime import datetime, timedelta
from random import randint, random, seed

from GAMealPlanner import GAMealPlanner
from pygad import GA

from models import (
    Ingredient,
    MealPlanningEnvironment,
    NutritionalInformation,
    Pantry,
    PantryIngredient,
    UserPreferences,
)

In [2]:
preferences = UserPreferences(
    weekly_budget=50,
    calorie_target_per_day=2500,
    protein_target_per_day=50,
    is_vegetarian=False,
    is_vegan=False,
    requires_gluten_free=False,
    requires_lactose_free=False,
)

In [3]:
all_ingredients = []

with open("../recipe_extraction/supplemented_structured_ingredients.json", "r") as f:
    ingredients_data = json.load(f)

    for ingredient_data in ingredients_data:
        ingredient_nutritional_information = NutritionalInformation(**ingredient_data["nutritional_information"])

        ingredient = Ingredient(
            name=ingredient_data["name"], nutritional_information=ingredient_nutritional_information
        )
        all_ingredients.append(ingredient)

In [4]:
def get_ingredient(ingredient_name: str) -> Ingredient | None:
    return next((ingredient for ingredient in all_ingredients if ingredient.name == ingredient_name), None)

In [5]:
seed(1)

In [6]:
def get_pantry_ingredient(
    ingredient_name: str, estimated_expiration_date: datetime, estimated_financial_cost: float
) -> PantryIngredient:
    ingredient = get_ingredient(ingredient_name)

    if ingredient is None:
        raise ValueError(f"Ingredient '{ingredient_name}' not found in ingredient list")

    return PantryIngredient(
        name=ingredient.name,
        nutritional_information=ingredient.nutritional_information,
        estimated_expiration_date=estimated_expiration_date,
        estimated_financial_cost=estimated_financial_cost,
    )

In [7]:
CURRENT_DATE = datetime.now()

In [8]:
pantry_ingredient_name_by_quantity = {
    "chicken breast": 200,
    "broccoli": 150,
    "rice": 100,
}

In [9]:
pantry_ingredients = [
    get_pantry_ingredient(ingredient_name, CURRENT_DATE + timedelta(days=randint(1, 7)), random())
    for ingredient_name, quantity in pantry_ingredient_name_by_quantity.items()
]

In [10]:
pantry_ingredient_by_quantity = dict(zip(pantry_ingredients, pantry_ingredient_name_by_quantity.values()))

In [11]:
pantry = Pantry()

for pantry_ingredient, quantity in pantry_ingredient_by_quantity.items():
    pantry.add(
        pantry_ingredient,
        quantity,
    )

In [12]:
meal_planning_environment = MealPlanningEnvironment(
    recipes=[],
    pantry=pantry,
    preferences=preferences,
)

In [13]:
meal_planning_environment.load_recipes_from_json("../recipe_extraction/supplemented_structured_recipes.json")

In [14]:
recipes = meal_planning_environment.recipes
pantry_stock = meal_planning_environment.pantry.stock
days_until_expiry = meal_planning_environment.pantry.get_days_until_expiry(CURRENT_DATE)
ingredient_costs = meal_planning_environment.pantry.ingredient_costs
preferences = meal_planning_environment.preferences

In [ ]:
planner = GAMealPlanner()


def fitness_function(_ga_instance, solution, _solution_index):
    return planner.evaluate_meal_plan(solution, recipes, pantry_stock, days_until_expiry, preferences, ingredient_costs)

In [17]:
GENERATION_PRINT_INTERVAL = 10

In [18]:
def on_generation(ga_instance):
    num_generations = ga_instance.generations_completed

    if num_generations % GENERATION_PRINT_INTERVAL == 0:
        best_fitness = ga_instance.best_solution()[1]
        print(f"Generation {num_generations}, Best Fitness: {best_fitness:.2f}")

In [19]:
NUM_GENERATIONS = 1000
NUM_PARENTS_MATING = 20
POPULATION_SIZE = 100
NUM_GENES = 21  # 3 meals/day x 7 days

In [20]:
ga_instance = GA(
    num_generations=NUM_GENERATIONS,
    num_parents_mating=NUM_PARENTS_MATING,
    fitness_func=fitness_function,
    sol_per_pop=POPULATION_SIZE,
    num_genes=NUM_GENES,
    gene_type=int,
    gene_space=list(range(len(meal_planning_environment.recipes))),
    parent_selection_type="tournament",
    K_tournament=5,
    keep_elitism=1,
    crossover_type="single_point",
    crossover_probability=0.8,
    mutation_type="random",
    mutation_probability=0.1,
    on_generation=on_generation,
    random_seed=1,
)

In [21]:
ga_instance.run()

Generation 10, Best Fitness: -48.55
Generation 20, Best Fitness: -33.24
Generation 30, Best Fitness: -24.84
Generation 40, Best Fitness: -17.95
Generation 50, Best Fitness: -17.95
Generation 60, Best Fitness: -17.70
Generation 70, Best Fitness: -16.02
Generation 80, Best Fitness: -14.69
Generation 90, Best Fitness: -13.95
Generation 100, Best Fitness: -10.40
Generation 110, Best Fitness: -10.40
Generation 120, Best Fitness: -9.52
Generation 130, Best Fitness: -9.52
Generation 140, Best Fitness: -8.78
Generation 150, Best Fitness: -8.16
Generation 160, Best Fitness: -8.16
Generation 170, Best Fitness: -8.16
Generation 180, Best Fitness: -8.16
Generation 190, Best Fitness: -7.12
Generation 200, Best Fitness: -7.12
Generation 210, Best Fitness: -6.97
Generation 220, Best Fitness: -5.93
Generation 230, Best Fitness: -5.93
Generation 240, Best Fitness: -5.70
Generation 250, Best Fitness: -5.70
Generation 260, Best Fitness: -5.70
Generation 270, Best Fitness: -5.70
Generation 280, Best Fitne

KeyboardInterrupt: 

In [22]:
best_meal_plan, best_fitness, _ = ga_instance.best_solution()

print(f"Best fitness score: {best_fitness:.2f}")

Best fitness score: 0.03


In [23]:
for day in range(7):
    print(f"\nDay {day + 1} meal plan:")

    for meal in range(3):
        recipe = meal_planning_environment.recipes[int(best_meal_plan[day * 3 + meal])]

        print(f"Meal {meal + 1}: {recipe.name}")


Day 1 meal plan:
Meal 1: Diner-Style Bacon for a Crowd
Meal 2: Herbed Oyster Crakers
Meal 3: Corn Bread

Day 2 meal plan:
Meal 1: Quick Pecan Tart
Meal 2: Peanut Butter Chocolate Chip Breads
Meal 3: Grand Marnier Brownie Kisses

Day 3 meal plan:
Meal 1: Thick Tahini Sauce
Meal 2: Roasted Broccoli Florets with Toasted Breadcrumb Gremolata
Meal 3: "Like a Caesar"

Day 4 meal plan:
Meal 1: Key Lime Pie
Meal 2: Tuna and Vegetable Fettuccine with Lemon Breadcrumbs
Meal 3: Zucchini Salad With Ajo Blanco Dressing & Spiced Nuts

Day 5 meal plan:
Meal 1: Peppered Lamb with Pine Nut Sauce
Meal 2: Red Bell Pepper Cream Sauce
Meal 3: Turkey Cream Puff Pie

Day 6 meal plan:
Meal 1: Chocolate “Creme Egg” Tart
Meal 2: Sicilian-Style Pasta with Sardines
Meal 3: Roast Lamb with Pistou

Day 7 meal plan:
Meal 1: Grasshopper II
Meal 2: Harissa Shrimp And Summer Vegetable Sauté
Meal 3: Plain Genoise


In [24]:
# going through the best meal plan and calculating how much of each ingredient is consumed from the pantry

consumed_stock = dict.fromkeys(pantry_stock.keys(), 0)

for index in best_meal_plan:
    recipe = recipes[int(index)]

    for ingredient_name, quantity_needed in recipe.ingredients.items():
        available = pantry_stock.get(ingredient_name, 0) - consumed_stock.get(ingredient_name, 0)
        from_pantry = max(0, min(available, quantity_needed))
        consumed_stock[ingredient_name] = consumed_stock.get(ingredient_name, 0) + from_pantry

In [25]:
import pandas as pd

pantry_data = []

for ingredient in pantry.ingredients:
    available = pantry_stock[ingredient.name]
    used = consumed_stock.get(ingredient.name, 0)
    unused = max(0, available - used)
    days = days_until_expiry[ingredient.name]
    expiry_str = str(days) + "d"
    pantry_data.append(
        {
            "Ingredient": ingredient.name,
            "Available": available,
            "Consumed": used,
            "Unused": unused,
            "Expires in": expiry_str,
        }
    )

pantry_data_df = pd.DataFrame(pantry_data)

pantry_data_df

,Ingredient,Available,Consumed,Unused,Expires in
0,chicken breast,200,0.0,200.0,2d
1,broccoli,150,150.0,0.0,7d
2,rice,100,88.0,12.0,3d


In [26]:
meal_plan = []

for day in range(7):
    meal_indices = best_meal_plan[day * 3 : day * 3 + 3]
    meal_names = [recipes[int(index)].name for index in meal_indices]
    meal_plan.append(
        {
            "Day": day + 1,
            "Meal 1": meal_names[0],
            "Meal 2": meal_names[1],
            "Meal 3": meal_names[2],
        }
    )

meal_plan_df = pd.DataFrame(meal_plan)

meal_plan_df

,Day,Meal 1,Meal 2,Meal 3
0,1,Diner-Style Bacon for a Crowd,Herbed Oyster Crakers,Corn Bread
1,2,Quick Pecan Tart,Peanut Butter Chocolate Chip Breads,Grand Marnier Brownie Kisses
2,3,Thick Tahini Sauce,Roasted Broccoli Florets with Toasted Breadcru...,"""Like a Caesar"""
3,4,Key Lime Pie,Tuna and Vegetable Fettuccine with Lemon Bread...,Zucchini Salad With Ajo Blanco Dressing & Spic...
4,5,Peppered Lamb with Pine Nut Sauce,Red Bell Pepper Cream Sauce,Turkey Cream Puff Pie
5,6,Chocolate “Creme Egg” Tart,Sicilian-Style Pasta with Sardines,Roast Lamb with Pistou
6,7,Grasshopper II,Harissa Shrimp And Summer Vegetable Sauté,Plain Genoise


In [27]:
shopping_list = {}

for index in best_meal_plan:
    recipe = recipes[int(index)]

    for ingredient_name, quantity_needed in recipe.ingredients.items():
        available = pantry_stock.get(ingredient_name, 0) - consumed_stock.get(ingredient_name, 0)
        to_buy = max(0, quantity_needed - available)
        if to_buy > 0:
            shopping_list[ingredient_name] = shopping_list.get(ingredient_name, 0.0) + to_buy

shopping_data = [
    {
        "Ingredient": name,
        "Quantity to Buy (g)": round(qty, 1),
        "Cost (€)": round((qty / 100.0) * ingredient_costs.get(name, 1.0), 2),
    }
    for name, qty in sorted(shopping_list.items())
]

shopping_df = pd.DataFrame(shopping_data)

total_cost = shopping_df["Cost (€)"].sum()
total_row = pd.DataFrame([{"Ingredient": "TOTAL", "Quantity to Buy (g)": "", "Cost (€)": round(total_cost, 2)}])
shopping_df = pd.concat([shopping_df, total_row], ignore_index=True)

print(f"Shopping list ({len(shopping_data)} ingredients, total cost: €{total_cost:.2f})\n")
shopping_df

Shopping list (139 ingredients, total cost: €50.65)



,Ingredient,Quantity to Buy (g),Cost (€)
0,(105-115ºF) water,118.3,1.18
1,"A 9"" tart pan with removable bottom",5.9,0.06
2,Fresh basil leaves,1.2,0.01
3,Grand Marnier,2.6,0.03
4,Ice cubes,50.0,0.50
...,...,...,...
135,whole milk,25.3,0.25
136,yellow food coloring,5.9,0.06
137,zucchini,1.8,0.02
138,zucchinis,8.0,0.08


In [28]:
nutrition_data = []

for day in range(7):
    meal_indices = best_meal_plan[day * 3 : day * 3 + 3]
    daily_calories = sum(recipes[int(i)].nutritional_information.calories or 0.0 for i in meal_indices)
    daily_protein = sum(recipes[int(i)].nutritional_information.protein or 0.0 for i in meal_indices)
    nutrition_data.append(
        {
            "Calories (kcal)": round(daily_calories, 1),
            "Δ Calories and Target Calories (kcal)": round(daily_calories - preferences.calorie_target_per_day, 1),
            "Protein (g)": round(daily_protein, 1),
            "Δ Protein and Target Protein (g)": round(daily_protein - preferences.protein_target_per_day, 1),
        }
    )

nutrition_df = pd.DataFrame(nutrition_data, index=["Day 1", "Day 2", "Day 3", "Day 4", "Day 5", "Day 6", "Day 7"])

nutrition_df

,Calories (kcal),Δ Calories and Target Calories (kcal),Protein (g),Δ Protein and Target Protein (g)
Day 1,2506.3,6.3,51.1,1.1
Day 2,2434.0,-66.0,50.6,0.6
Day 3,2496.3,-3.7,55.1,5.1
Day 4,2509.8,9.8,48.5,-1.5
Day 5,2494.4,-5.6,57.2,7.2
Day 6,2518.3,18.3,51.4,1.4
Day 7,2538.7,38.7,51.1,1.1
